# **Beds Data Cleaning**

## 1. Importing Libraries and Loading Data

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR/"04_beds_dirty.csv")

## 2. Initial Data Inspection

In [3]:
df.head()

,Bed_ID,Department_ID,Ward_Type,Floor,Bed_Status,Daily_Rate
0,B1000,D005,General Ward,1,NaN,6354
1,B1001,D011,Emergency,5,NaN,6150
2,B1002,D012,Private,5,OCCUPIED,2477
3,B1003,D003,NICU,2,NaN,11467
4,B1004,D006,Isolation,5,Occupied,6149


In [9]:
df.shape

(700, 6)

In [10]:
df.columns.tolist()

['Bed_ID', 'Department_ID', 'Ward_Type', 'Floor', 'Bed_Status', 'Daily_Rate']

In [5]:
df.dtypes

Bed_ID           object
Department_ID    object
Ward_Type        object
Floor             int64
Bed_Status       object
Daily_Rate        int64
dtype: object

In [6]:
df.isna().sum()

Bed_ID             0
Department_ID      0
Ward_Type          0
Floor              0
Bed_Status       130
Daily_Rate         0
dtype: int64

In [7]:
df.duplicated().sum()

0

## 3. Audit of Specific Columns

In [11]:
df["Bed_ID"].duplicated().sum()

0

In [12]:
df["Department_ID"].unique()


array(['D005', 'D011', 'D012', 'D003', 'D006', 'D013', 'D016', 'D018',
       'D015', 'D004', 'D008', 'D014', 'D001', 'D002', 'D009', 'D010',
       'D007', 'D017'], dtype=object)

In [13]:
df["Ward_Type"].unique()


array(['General Ward', 'Emergency', 'Private', 'NICU', 'Isolation',
       'Semi Private', 'ICU'], dtype=object)

In [14]:
df["Floor"].unique()


array([1, 5, 2, 0, 4, 3], dtype=int64)

In [15]:
df["Bed_Status"].value_counts(dropna=False)

Bed_Status
NaN            130
OCCUPIED       130
Maintenance    118
Occupied       115
Available      106
available      101
Name: count, dtype: int64

In [17]:
df["Daily_Rate"].describe()

count      700.000000
mean      6086.514286
std       3517.459802
min          0.000000
25%       3079.750000
50%       6177.000000
75%       9116.000000
max      11993.000000
Name: Daily_Rate, dtype: float64

In [18]:
(df["Daily_Rate"] <= 0).sum()

25

In [19]:
df.loc[df["Daily_Rate"] <= 0, "Daily_Rate"].value_counts()

Daily_Rate
0    25
Name: count, dtype: int64

In [20]:
departments_df = pd.read_csv(CLEANED_DIR / "02_departments_cleaned.csv")

In [22]:
valid_department_ids = departments_df["Department_ID"].dropna().unique()

In [25]:
df.loc[~df["Department_ID"].isin(valid_department_ids), "Department_ID"]

Series([], Name: Department_ID, dtype: object)

## 4. Data Cleaning

### 4.1 Bed_Status

In [26]:
bed_status_cleaned = (df["Bed_Status"].str.strip().str.title())

In [27]:
bed_status_cleaned.value_counts(dropna=False)

Bed_Status
Occupied       245
Available      207
NaN            130
Maintenance    118
Name: count, dtype: int64

In [28]:
df["Bed_Status"] = bed_status_cleaned

### 4.2 Daily_Rate

In [29]:
daily_rate_cleaned = df["Daily_Rate"].mask(df["Daily_Rate"] <= 0)

In [30]:
daily_rate_cleaned.isna().sum()

25

In [31]:
df["Daily_Rate"] = daily_rate_cleaned

## 5. Final Validation

In [32]:
df.shape

(700, 6)

In [33]:
df.dtypes

Bed_ID            object
Department_ID     object
Ward_Type         object
Floor              int64
Bed_Status        object
Daily_Rate       float64
dtype: object

In [35]:
df.isna().sum()

Bed_ID             0
Department_ID      0
Ward_Type          0
Floor              0
Bed_Status       130
Daily_Rate        25
dtype: int64

In [37]:
df.duplicated().sum()

0

In [38]:
df["Bed_ID"].duplicated().sum()

0

In [40]:
df["Department_ID"].unique()

array(['D005', 'D011', 'D012', 'D003', 'D006', 'D013', 'D016', 'D018',
       'D015', 'D004', 'D008', 'D014', 'D001', 'D002', 'D009', 'D010',
       'D007', 'D017'], dtype=object)

In [42]:
df["Ward_Type"].unique()

array(['General Ward', 'Emergency', 'Private', 'NICU', 'Isolation',
       'Semi Private', 'ICU'], dtype=object)

In [43]:
df["Floor"].unique()

array([1, 5, 2, 0, 4, 3], dtype=int64)

In [44]:
df["Bed_Status"].unique()

array([nan, 'Occupied', 'Available', 'Maintenance'], dtype=object)

In [45]:
df["Daily_Rate"].min()

545.0

## 6. Export Cleaned Data

In [46]:
df.to_csv(CLEANED_DIR / "04_beds_cleaned.csv", index=False)